# HyperBird Processing Pipeline

Batch processing pipeline for HyperBird HSI cubes.
- Reads ENVI (.hdr/.raw) from an input folder.
- Creates a sibling "{input_name}_processed" folder with per-image subfolders.
- Builds/updates white/black row-reference Zarrs when encountering files named like "xxx-white*" / "xxx-black*".
- Applies flat-field correction (FFC) to sample cubes using current refs.
- Converts corrected cube to RGB, runs GSAM2 segmentation to obtain ROI masks.
- Computes ROI mean/std spectra and saves CSV per image.
- Cleans up temp Zarr files after each image; white/black refs are deleted only when a newer ref replaces them or at the very end.
- Provides a helper to merge all per-image CSVs into one.

## Enviroment Setup

In [1]:
import os
import sys
import time
import cv2
import torch
import csv
import json
import shutil
import threading
import torch
import copy
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from typing import Optional, List, Tuple, Set
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import datetime
import subprocess
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.envi_zarr_conversion import convert_envi_to_zarr, convert_zarr_to_envi
from src.hsi2color_zarr import hsi_to_color_zarr
from src.reference_building_zarr import (
    build_white_reference_zarr,
    build_black_reference_zarr,
)
from src.flat_field_correction_zarr import flat_field_correction_zarr
from src.roi_mean_spectra_zarr import roi_mean_spectra_zarr
from sam2.gsam2_segmenter import GSAM2_Segmenter

In [2]:
# Check CUDA support 
!nvcc --version
!nvidia-smi

# Determine the device to use
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Print device information
if DEVICE.type == "cuda":
    print("Available Device: GPU")
    print(f"Device Name: {torch.cuda.get_device_name(DEVICE)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Capability (SM version): {torch.cuda.get_device_capability(DEVICE)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(DEVICE)} bytes")
    print(f"Memory Cached: {torch.cuda.memory_reserved(DEVICE)} bytes")
else:
    print("Available Device: CPU")

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Jan_15_19:20:09_PST_2025
Cuda compilation tools, release 12.8, V12.8.61
Build cuda_12.8.r12.8/compiler.35404655_0
Wed Jan 14 16:30:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...    Off |   00000000:01:00.0  On |                  N/A |
|  0%   37C    P8       

## Helpers

In [3]:
_ENVI_HDR_EXT = ".hdr"
_ENVI_RAW_EXT = ".raw"


def _numeric_prefix(fname: str) -> Optional[int]:
    """Return the first three chars as integer if they are digits, else None."""
    base = os.path.basename(fname)
    if len(base) >= 3 and base[:3].isdigit():
        return int(base[:3])
    return None


def _is_white(name: str) -> bool:
    return "white" in name.lower()


def _is_black(name: str) -> bool:
    return "black" in name.lower() or "dark" in name.lower()


def _ensure_dir(p: str) -> None:
    os.makedirs(p, exist_ok=True)


def _pair_envi_files(folder: str) -> List[str]:
    """Return sorted list of .hdr files that have a matching .raw file, ordered by 001..999 prefix."""
    hdrs = []
    for fn in os.listdir(folder):
        if fn.lower().endswith(_ENVI_HDR_EXT):
            raw = os.path.splitext(fn)[0] + _ENVI_RAW_EXT
            if os.path.exists(os.path.join(folder, raw)):
                hdrs.append(fn)
    # sort by 3-digit prefix; keep stable order for ties by filename
    hdrs.sort(
        key=lambda x: (
            _numeric_prefix(x) if _numeric_prefix(x) is not None else 10_000,
            x.lower(),
        )
    )
    return [os.path.join(folder, h) for h in hdrs]


def _write_image(path: str, img_rgb: np.ndarray, *, autoscale=False) -> None:
    """
    Save an RGB image:
      - handles float arrays (0..1 or arbitrary) → uint8
      - drops alpha channel if present
      - optional percentile autoscaling to avoid all-black images
    """
    _ensure_dir(os.path.dirname(path))

    x = np.asarray(img_rgb)

    # Ensure 3 channels (drop alpha, expand gray)
    if x.ndim == 2:
        x = np.stack([x, x, x], axis=-1)
    elif x.ndim == 3 and x.shape[-1] == 4:
        x = x[..., :3]

    # Convert to uint8
    if np.issubdtype(x.dtype, np.floating):
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        if autoscale:
            lo = np.percentile(x, 1)
            hi = np.percentile(x, 99)
            if hi > lo:
                x = (x - lo) / (hi - lo)
        # assume 0..1 after optional autoscale; clamp and scale
        x = np.clip(x, 0.0, 1.0)
        x = (x * 255.0 + 0.5).astype(np.uint8, copy=False)
    else:
        x = np.clip(x, 0, 255).astype(np.uint8, copy=False)

    # Write as PNG (OpenCV expects BGR)
    bgr = cv2.cvtColor(np.ascontiguousarray(x), cv2.COLOR_RGB2BGR)
    cv2.imwrite(path, bgr)


def _write_mask(path: str, mask_u8: np.ndarray) -> None:
    _ensure_dir(os.path.dirname(path))
    cv2.imwrite(path, mask_u8)


def _to_uint8_rgb(img):
    import numpy as np

    x = np.asarray(img)

    # Channel-first -> channel-last
    if x.ndim == 3 and x.shape[0] in (3, 4) and x.shape[-1] not in (3, 4):
        x = np.moveaxis(x, 0, -1)

    # Grayscale -> 3-channel
    if x.ndim == 2:
        x = np.stack([x, x, x], axis=-1)

    # Drop alpha if present
    if x.ndim == 3 and x.shape[-1] == 4:
        x = x[..., :3]

    # Convert dtype/range to uint8 [0,255]
    if np.issubdtype(x.dtype, np.floating):
        # If already 0..1, scale to 0..255; otherwise clip to 0..255
        x_min, x_max = float(np.nanmin(x)), float(np.nanmax(x))
        if 0.0 <= x_min and x_max <= 1.0:
            x = x * 255.0
        x = np.clip(x, 0.0, 255.0).astype(np.uint8, copy=False)
    else:
        x = np.clip(x, 0, 255).astype(np.uint8, copy=False)

    return np.ascontiguousarray(x)


def _safe_remove(p: str) -> None:
    try:
        if os.path.isdir(p):
            shutil.rmtree(p)
        elif os.path.exists(p):
            os.remove(p)
    except Exception:
        pass


def _prune_temp_workspace(temp_root: str, keep_paths: tuple[str, ...] = ()) -> None:
    """
    Delete EVERYTHING under temp_root except the paths listed in keep_paths
    (and their children). Use this at the end of each loop iteration.
    """
    if not os.path.isdir(temp_root):
        return
    keep_abs = {os.path.abspath(k) for k in keep_paths if k}
    for name in os.listdir(temp_root):
        p = os.path.abspath(os.path.join(temp_root, name))
        # preserve refs; delete everything else
        if any(p == k or p.startswith(k + os.sep) for k in keep_abs):
            continue
        _safe_remove(p)


def _save_roi_mean_std_plot(
    wavelengths, means_list, stds_list, ids=None, title=None, save_path=None, show=False
):
    import numpy as np
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    wl = np.asarray(wavelengths, dtype=float)
    if ids is None:
        ids = range(1, len(means_list) + 1)

    fig, ax = plt.subplots(figsize=(12, 8), dpi=300)
    for rid, mu, sd in zip(ids, means_list, stds_list):
        mu = np.asarray(mu, dtype=float)
        sd = np.asarray(sd, dtype=float)
        (line,) = ax.plot(wl, mu, label=f"ROI {rid}")
        ax.fill_between(
            wl, mu - sd, mu + sd, alpha=0.2, facecolor=line.get_color(), linewidth=0
        )

    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Intensity (a.u.)")
    if title:
        ax.set_title(title)
    ax.legend()
    # ax.grid(True)

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)


# Plotting for 1-row white/black reference
def _plot_reference_zarr(zarr_path: str, save_path: str, title: str):
    """
    Plot mean ±1σ spectrum from a 1-row white/black reference Zarr.
    - No relative imports (works when run as a script).
    - Robust to BIL/BIP/BSQ: detects the bands axis by matching wavelengths length.
    """
    import numpy as np
    import zarr

    ref = zarr.open(zarr_path, mode="r")
    arr = ref if isinstance(ref, zarr.Array) else ref[list(ref.array_keys())[0]]
    attrs = arr.attrs.asdict() if hasattr(arr.attrs, "asdict") else dict(arr.attrs)

    # Wavelengths
    wavelengths = attrs.get("wavelength", None)
    if wavelengths is None:
        raise ValueError(f"Missing 'wavelength' in attrs for {zarr_path}")
    wl = np.asarray(wavelengths, dtype=float).ravel()
    n_bands = wl.size

    x = np.asarray(arr[...], dtype=float)
    if x.ndim != 3:
        raise ValueError(f"Expected 3D reference cube; got shape {x.shape}")

    # Find the bands axis by matching n_bands
    band_axes = [i for i, s in enumerate(x.shape) if s == n_bands]
    if not band_axes:
        # Fallback: if none match exactly, pick the axis whose size is closest to n_bands
        band_axis = int(np.argmin([abs(s - n_bands) for s in x.shape]))
    else:
        # Prefer the last matching axis (common in BIP)
        band_axis = band_axes[-1]

    # Move bands axis to last → (R, C, B)
    x_bands_last = np.moveaxis(x, band_axis, -1)

    # Flatten spatial dims → (pixels, bands)
    pixels_by_bands = x_bands_last.reshape(-1, x_bands_last.shape[-1])

    # Mean/Std over pixels
    mu = np.nanmean(pixels_by_bands, axis=0)
    sd = np.nanstd(pixels_by_bands, axis=0)

    # Guard against length mismatch (prevents: x and y must have same first dimension)
    if mu.shape[0] != n_bands:
        # If mismatch persists, align by min length
        m = min(mu.shape[0], n_bands)
        wl_aligned = wl[:m]
        mu = mu[:m]
        sd = sd[:m]
    else:
        wl_aligned = wl

    # Use existing util to save the plot
    _save_roi_mean_std_plot(
        wl_aligned,
        [mu],
        [sd],
        ids=["Ref"],
        title=title,
        save_path=save_path,
        show=False,
    )


def _save_csv(
    path: str, wavelengths, roi_means_rows, *, ids=None, image_name: str = ""
):
    """
    Wide CSV per image:
      - First column = 'image'
      - Second column = 'id' (ROI order)
      - Remaining columns = wavelengths (means only)
    """
    _ensure_dir(os.path.dirname(path))
    header = ["image", "id"] + [str(float(w)) for w in wavelengths]
    import csv

    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        if ids is None:
            ids = range(1, len(roi_means_rows) + 1)
        for rid, row in zip(ids, roi_means_rows):
            w.writerow([image_name, int(rid)] + [float(x) for x in row])


def _merge_csvs(processed_root: str, out_csv: str):
    """
    Merge the single CSV found in each immediate subfolder of `processed_root`.
    Expect per-file header: ['image','id', <band1_nm>, <band2_nm>, ...]
    """
    import os, csv

    processed_root = os.path.abspath(processed_root)
    rows = []
    header_wavelengths = None

    for name in sorted(os.listdir(processed_root)):
        sub = os.path.join(processed_root, name)
        if not os.path.isdir(sub):
            continue

        csv_files = [f for f in os.listdir(sub) if f.lower().endswith(".csv")]
        if not csv_files:
            continue
        if len(csv_files) != 1:
            raise RuntimeError(f"Expected exactly one CSV in {sub}, found: {csv_files}")

        csv_path = os.path.join(sub, csv_files[0])
        with open(csv_path, "r", newline="") as f:
            r = csv.reader(f)
            try:
                header = next(r)  # expect ['image', 'id', '400.0', ...]
            except StopIteration:
                continue

            if not (
                header
                and header[0] == "image"
                and header[1] == "id"
                and len(header) >= 3
            ):
                raise RuntimeError(f"CSV format mismatch in {csv_path}")

            current_wl = header[2:]
            if header_wavelengths is None:
                header_wavelengths = current_wl
            elif header_wavelengths != current_wl:
                raise RuntimeError(f"Wavelength header mismatch in {csv_path}")

            for row in r:
                if row:
                    rows.append(row)  # already [image, id, ...values]

    if not rows or header_wavelengths is None:
        raise RuntimeError(f"No CSVs found under {processed_root}")

    os.makedirs(os.path.dirname(out_csv) or ".", exist_ok=True)
    with open(out_csv, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["image", "id"] + header_wavelengths)
        w.writerows(rows)


def _shrink_masks(masks: List[np.ndarray], shrink_pixels: int = 50) -> List[np.ndarray]:
    """
    Shrink each mask by eroding the edges by shrink_pixels.

    Args:
        masks: List of mask arrays (can be bool, uint8, or float)
        shrink_pixels: Number of pixels to shrink from each edge

    Returns:
        List of shrunk masks with same dtype as input
    """
    if shrink_pixels <= 0:
        return masks

    # Create circular kernel for erosion
    kernel_size = shrink_pixels * 2 + 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))

    shrunk_masks = []
    for mask in masks:
        # Convert to binary uint8 if needed
        if mask.dtype == bool:
            mask_u8 = mask.astype(np.uint8) * 255
        elif np.issubdtype(mask.dtype, np.floating):
            mask_u8 = (mask > 0.5).astype(np.uint8) * 255
        else:
            mask_u8 = (mask > 0).astype(np.uint8) * 255

        # Erode the mask
        eroded = cv2.erode(mask_u8, kernel, iterations=1)

        # Convert back to original dtype
        if mask.dtype == bool:
            shrunk_mask = (eroded > 0).astype(bool)
        elif np.issubdtype(mask.dtype, np.floating):
            shrunk_mask = (eroded > 0).astype(mask.dtype)
        else:
            shrunk_mask = (eroded > 0).astype(mask.dtype)

        shrunk_masks.append(shrunk_mask)

    return shrunk_masks


def _save_masks_overlay(
    image: np.ndarray,
    masks: list[np.ndarray],
    color: tuple[int, int, int] = (128, 0, 255),  # Purple in RGB
    opacity: float = 0.3,
    output_path: str = "overlay.png",
) -> np.ndarray:
    """
    Overlays a list of binary masks on an RGB image and saves the result as a PNG.

    Args:
        image (np.ndarray): RGB image (H, W, 3), dtype=uint8 or float.
        masks (list[np.ndarray]): List of binary or integer masks (H, W), nonzero = region to overlay.
        color (tuple[int, int, int]): Overlay color (R, G, B).
        opacity (float): Blend factor between 0 (transparent) and 1 (opaque).
        output_path (str): File path to save the resulting PNG.
    Returns:
        np.ndarray: The resulting RGB image with all overlays applied.
    """
    # Ensure correct shape
    assert image.ndim == 3 and image.shape[2] == 3, "Input image must be RGB (H, W, 3)"
    assert (
        isinstance(masks, (list, tuple)) and len(masks) > 0
    ), "masks must be a non-empty list or tuple"
    H, W = image.shape[:2]
    for i, mask in enumerate(masks):
        assert mask.ndim == 2, f"Mask {i} must be 2D (H, W)"
        assert mask.shape == (H, W), f"Mask {i} shape does not match image"

    # Defensive conversion: Fix all-black (or all-zero) RGB image
    img = image
    # If image is float, convert to uint8 scaling up
    if np.issubdtype(img.dtype, np.floating):
        # Try normalization if image is almost all zeros
        vmin, vmax = img.min(), img.max()
        if vmax - vmin > 1e-6:
            img = 255 * (img - vmin) / (vmax - vmin)
        img = np.clip(img, 0, 255).astype(np.uint8)
    if not np.issubdtype(img.dtype, np.uint8):
        img = np.clip(img, 0, 255).astype(np.uint8)

    # If image is (nearly) all black, brighten for visualization
    if np.median(img) < 5:
        # Try simple stretching of contrast
        p_low, p_high = np.percentile(img, 1), np.percentile(img, 99)
        if p_high > p_low:
            img = 255 * (img - p_low) / (p_high - p_low)
            img = np.clip(img, 0, 255).astype(np.uint8)
        # Otherwise boost all pixels equally
        else:
            img = np.full_like(img, 128, dtype=np.uint8)

    # Convert RGB to BGR for OpenCV
    image_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    overlay = image_bgr.copy()
    bgr_color = color[::-1]  # (R,G,B) -> (B,G,R)

    # Apply overlay color to masked regions
    for mask in masks:
        if not np.any(mask):
            continue
        overlay[mask > 0] = bgr_color

    # Blend overlay
    blended = cv2.addWeighted(overlay, opacity, image_bgr, 1 - opacity, 0)

    # Convert back to RGB for saving
    blended_rgb = cv2.cvtColor(blended, cv2.COLOR_BGR2RGB)

    # Save output as PNG
    cv2.imwrite(output_path, cv2.cvtColor(blended_rgb, cv2.COLOR_RGB2BGR))
    return blended_rgb

## Processing Pipeline

In [4]:
@dataclass
class PipelineState:
    processed_root: str
    white_ref_zarr: Optional[str] = None
    black_ref_zarr: Optional[str] = None
    stale_white_paths: List[str] = field(default_factory=list)
    stale_black_paths: List[str] = field(default_factory=list)


@dataclass
class PipelineCfg:
    temp_root: str
    save_ffc: bool
    assume_interleave: Optional[str]
    illuminants_mat_path: str
    illuminant_d: int
    shrink_distance: int


@dataclass
class SegmenterConfig:
    text_prompt: str
    hf_model_id: str
    detector_device: str
    sam2_cfg_path: str
    sam2_ckpt_path: str
    sam2_device: str
    box_threshold: float
    max_dets: int
    multimask_output: bool


# -------- single-image processor --------
def process_hdr(
    hdr_path: str,
    seg: GSAM2_Segmenter,
    seg_cfg: SegmenterConfig,
    pipeline_cfg: PipelineCfg,
    state: PipelineState,
    *,
    input_folder_name_for_meta: str,
    overwrite: bool = True,
) -> Optional[str]:
    """
    Process exactly one ENVI pair (hdr+raw) pointed to by `hdr_path`.
    """
    base = os.path.splitext(os.path.basename(hdr_path))[0]  # e.g., "001-white"
    is_white = _is_white(base)
    is_black = _is_black(base)

    # Per-image output folder
    out_dir = os.path.join(state.processed_root, base)
    if os.path.exists(out_dir):
        if not overwrite:
            return None  # skip already-processed
    _ensure_dir(out_dir)

    # Convert to Zarr (temp)
    _ensure_dir(pipeline_cfg.temp_root)
    zarr_path = os.path.join(pipeline_cfg.temp_root, f"{base}.zarr")
    convert_envi_to_zarr(hdr_path, zarr_path, chunks=(256, 256, 256), overwrite=True)

    # --- WHITE reference ---
    if is_white:
        rowref_path = os.path.join(pipeline_cfg.temp_root, f"{base}_rowref_white.zarr")
        white_ret = build_white_reference_zarr(
            white_zarr_path=zarr_path,
            illuminants_path=pipeline_cfg.illuminants_mat_path,
            interleave=pipeline_cfg.assume_interleave,
            normalize="global",
            out_rowref_zarr=rowref_path,
            overwrite=True,
        )
        if state.white_ref_zarr and os.path.abspath(
            state.white_ref_zarr
        ) != os.path.abspath(rowref_path):
            state.stale_white_paths.append(state.white_ref_zarr)
        state.white_ref_zarr = rowref_path

        # Save returned RGB + mask (best-effort)
        try:
            rgb = white_ret.get("rgb", None)
            if rgb is not None:
                _write_image(os.path.join(out_dir, base + "_rgb.png"), rgb)
            mask = white_ret.get("mask", None)
            if mask is not None:
                mask_u8 = (
                    (mask.astype(np.uint8) * 255) if mask.dtype != np.uint8 else mask
                )
                _write_mask(os.path.join(out_dir, base + "_mask.png"), mask_u8)
        except Exception:
            pass

        # Plot white reference spectrum
        _plot_reference_zarr(
            zarr_path=rowref_path,
            save_path=os.path.join(out_dir, f"{base}_white_ref_plot.svg"),
            title="White Reference Mean ±1σ",
        )

        # Export row-ref white as ENVI
        convert_zarr_to_envi(
            state.white_ref_zarr, os.path.join(out_dir, f"{base}_rowref.raw")
        )

        # cleanup temp cube
        try:
            os.remove(zarr_path)
        except Exception:
            pass
        return out_dir

    # --- BLACK reference ---
    if is_black:
        rowref_path = os.path.join(pipeline_cfg.temp_root, f"{base}_rowref_black.zarr")
        build_black_reference_zarr(
            zarr_path,
            rowref_path,
            interleave=pipeline_cfg.assume_interleave,
            overwrite=True,
        )
        if state.black_ref_zarr and os.path.abspath(
            state.black_ref_zarr
        ) != os.path.abspath(rowref_path):
            state.stale_black_paths.append(state.black_ref_zarr)
        state.black_ref_zarr = rowref_path

        # Plot black reference spectrum
        _plot_reference_zarr(
            zarr_path=rowref_path,
            save_path=os.path.join(out_dir, f"{base}_black_ref_plot.svg"),
            title="Black Reference Mean ±1σ",
        )

        # Export row-ref black as ENVI
        convert_zarr_to_envi(
            state.black_ref_zarr, os.path.join(out_dir, f"{base}_rowref.raw")
        )

        try:
            os.remove(zarr_path)
        except Exception:
            pass
        return out_dir

    # --- SAMPLE: require refs first ---
    if not state.white_ref_zarr or not state.black_ref_zarr:
        try:
            os.remove(zarr_path)
        finally:
            raise RuntimeError(
                f"Missing reference(s) before processing sample '{base}'. "
                f"white_ref_zarr={state.white_ref_zarr}, black_ref_zarr={state.black_ref_zarr}"
            )

    # 1) FFC -> Zarr
    ffc_zarr = os.path.join(pipeline_cfg.temp_root, f"{base}_ffc.zarr")
    flat_field_correction_zarr(
        zarr_path,
        state.white_ref_zarr,
        state.black_ref_zarr,
        ffc_zarr,
        interleave=pipeline_cfg.assume_interleave,
    )
    if pipeline_cfg.save_ffc:
        convert_zarr_to_envi(ffc_zarr, os.path.join(out_dir, f"{base}_ffc.raw"))

    # 2) HSI -> RGB
    rgb = hsi_to_color_zarr(
        ffc_zarr,
        pipeline_cfg.illuminants_mat_path,
        illuminant_d=pipeline_cfg.illuminant_d,
        assume_interleave=pipeline_cfg.assume_interleave,
        normalize="none",
        exposure=1.0,
        compute=True,
    )
    _write_image(os.path.join(out_dir, base + "_rgb.png"), rgb)

    # 3) Segment to get masks
    masks, annotated_bgr, mask_images, stacked_mask = seg.segment(
        rgb=_to_uint8_rgb(rgb), text_prompt=seg_cfg.text_prompt, visualize=False
    )

    # Shrink masks
    masks = _shrink_masks(masks, pipeline_cfg.shrink_distance)
    np.save(os.path.join(out_dir, base + "masks.npy"), masks)

    # Save annotations/masks
    cv2.imwrite(os.path.join(out_dir, base + "annotated.png"), annotated_bgr)
    stacked_mask = np.any(masks, axis=0).astype(np.uint8) * 255
    _write_mask(os.path.join(out_dir, base + "_mask_stacked.png"), stacked_mask)
    _save_masks_overlay(
        image=rgb,
        masks=masks,
        output_path=os.path.join(out_dir, base + "_mask_overlay.png"),
    )

    # 4) ROI mean spectra
    roi_means_rows, roi_stds_rows, roi_ids = [], [], []
    wavelengths_ref = None
    for idx, m in enumerate(masks, start=1):
        roi_mask = (m > 0) if m.dtype != bool else m
        lam, mu, sd = roi_mean_spectra_zarr(
            mask=roi_mask, zarr_path=ffc_zarr, interleave=pipeline_cfg.assume_interleave
        )
        if wavelengths_ref is None:
            wavelengths_ref = lam
        roi_means_rows.append(mu)
        roi_stds_rows.append(sd)
        roi_ids.append(idx)

    _save_csv(
        os.path.join(out_dir, base + "_roi_means.csv"),
        wavelengths_ref,
        roi_means_rows,
        ids=roi_ids,
        image_name=f"{input_folder_name_for_meta}_{base}",
    )
    _save_roi_mean_std_plot(
        wavelengths_ref,
        roi_means_rows,
        roi_stds_rows,
        ids=roi_ids,
        title="ROI Mean ±1σ",
        save_path=os.path.join(out_dir, base + "_roi_mean_std.svg"),
        show=False,
    )

    # 5) cleanup temp zarrs for this image (keep refs alive)
    _prune_temp_workspace(
        pipeline_cfg.temp_root, keep_paths=(state.white_ref_zarr, state.black_ref_zarr)
    )

    # 6) persist metadata
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(
            {
                "source_hdr": hdr_path,
                "order_index": _numeric_prefix(base),
                "white_ref_zarr": state.white_ref_zarr,
                "black_ref_zarr": state.black_ref_zarr,
                "prompt": seg_cfg.text_prompt,
                "hf_model_id": seg_cfg.hf_model_id,
                "sam2_cfg_path": seg_cfg.sam2_cfg_path,
                "sam2_ckpt_path": seg_cfg.sam2_ckpt_path,
                "box_threshold": seg_cfg.box_threshold,
                "max_dets": seg_cfg.max_dets,
            },
            f,
            indent=2,
        )

In [5]:
def process_folder(
    input_folder: str,
    seg: GSAM2_Segmenter,
    seg_cfg: SegmenterConfig,
    pipeline_cfg: PipelineCfg,
) -> str:
    """
    Run the batch pipeline on all ENVI pairs in `input_folder`.
    """
    input_folder = os.path.abspath(input_folder)
    parent = os.path.dirname(input_folder.rstrip(os.sep))
    in_name = os.path.basename(input_folder.rstrip(os.sep))
    processed_root = os.path.join(parent, f"{in_name}_processed")
    _ensure_dir(processed_root)

    state = PipelineState(processed_root=processed_root)

    _ensure_dir(pipeline_cfg.temp_root)

    hdr_list = _pair_envi_files(input_folder)
    if not hdr_list:
        raise RuntimeError(f"No ENVI .hdr/.raw pairs found in {input_folder}")

    for hdr_path in tqdm(
        hdr_list, desc=f"Processing images in {input_folder}", unit="image"
    ):
        try:
            process_hdr(
                hdr_path,
                seg=seg,
                seg_cfg=seg_cfg,
                pipeline_cfg=pipeline_cfg,
                state=state,
                input_folder_name_for_meta=os.path.basename(input_folder),
            )
        except Exception as e:
            # Optional: log and continue or re-raise depending on your tolerance
            print(f"[WARN] Failed to process {hdr_path}: {e}")

    # After loop: prune temp (allow refs to be cleaned now)
    try:
        _prune_temp_workspace(pipeline_cfg.temp_root)
    except Exception as e:
        print(f"Warning: Failed to prune temp workspace: {e}")

    # Merge all CSVs
    try:
        _merge_csvs(
            processed_root, os.path.join(processed_root, f"{in_name}_all_roi_means.csv")
        )
    except Exception as e:
        print(f"Warning: Failed to merge CSVs: {e}")

    # Remove temp folder (best-effort)
    try:
        os.rmdir(pipeline_cfg.temp_root)
    except Exception:
        pass

    return processed_root

## HyperBirdWatcher

In [6]:
"""
HyperBirdWatcher (always-on)

- Continuously watches WATCH_DIR for newly-added subfolders.
- For each new subfolder:
    - Spawns a monitor that looks for matching .hdr + .raw pairs.
    - Calls your `process_hdr(...)` for each new pair.
    - Writes outputs to "<folder>/_processed".
- Top-level watcher ignores folders that look like outputs or hidden
  (e.g., "_something", ".something", "*_processed", "*_export").
"""

# ---------- utilities ----------


def _now() -> str:
    import time as _t

    return _t.strftime("%Y-%m-%d %H:%M:%S", _t.localtime())


def _log(msg: str):
    print(f"{_now()} - {msg}")


def _ignore_top_level_folder(name: str) -> bool:
    """Filter out non-source top-level dirs (outputs/hidden/temp)."""
    n = (name or "").strip()
    return (
        not n
        or n.startswith(".")
        or n.startswith("_")
        or n.endswith("_processed")
        or n.endswith("_export")
    )


# Return a LIST (ordered) instead of a SET (unordered).
def _find_matching_pairs_in_dir(dirpath: str) -> List[Tuple[str, str]]:
    """Return list of (hdr_path, raw_path) pairs matched by basename (case-insensitive), ordered by numeric prefix."""
    try:
        files = os.listdir(dirpath)
    except FileNotFoundError:
        return []

    hdrs, raws = {}, {}
    for f in files:
        lf = f.lower()
        full = os.path.join(dirpath, f)
        if lf.endswith(".hdr"):
            hdrs[lf[:-4]] = full
        elif lf.endswith(".raw"):
            raws[lf[:-4]] = full

    matched: List[Tuple[str, str]] = [
        (hdrs[k], raws[k]) for k in hdrs.keys() & raws.keys()
    ]

    def _numeric_prefix(name: str) -> Optional[int]:
        import re

        m = re.match(r"^(\d{1,3})", name)
        return int(m.group(1)) if m else None

    matched.sort(
        key=lambda pr: (
            _numeric_prefix(os.path.basename(pr[0])) or 10_000,
            os.path.basename(pr[0]).lower(),
        )
    )
    return matched


# ---------- per-folder monitor ----------


class _FolderMonitor(threading.Thread):
    """Monitors one folder, detects new (.hdr,.raw) pairs, runs process_hdr for each."""

    def __init__(
        self,
        folder_path: str,
        stop_event: threading.Event,
        *,
        seg,
        seg_cfg,
        pipeline_cfg,
        file_poll_interval: float = 1.0,
    ):
        super().__init__(daemon=True)
        self.folder_path = folder_path
        self.stop_event = stop_event
        self.seg = seg
        self.seg_cfg = seg_cfg
        self.pipeline_cfg = pipeline_cfg
        self.file_poll_interval = file_poll_interval
        self.processed = set()

        # Add datetime to temp_root
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.pipeline_cfg.temp_root = os.path.join(
            self.pipeline_cfg.temp_root, f"{timestamp}"
        )

        # Put outputs *inside* the folder so the top-level watcher won't see them
        self.out_dir = os.path.join(self.folder_path, "_processed")
        os.makedirs(self.out_dir, exist_ok=True)

        # Fresh state for this folder/run
        self.state = PipelineState(processed_root=self.out_dir)

    def run(self):
        _log(f"Started monitoring folder: {self.folder_path}")
        folder_name_for_meta = os.path.basename(self.folder_path.rstrip(os.sep))

        while not self.stop_event.is_set():
            try:
                pairs = _find_matching_pairs_in_dir(self.folder_path)
            except FileNotFoundError:
                _log(f"Folder disappeared: {self.folder_path}. Exiting monitor.")
                return

            for hdr_path, raw_path in pairs:
                base = os.path.splitext(os.path.basename(hdr_path))[0].lower()
                if base in self.processed:
                    continue

                # simple stability check (files not growing)
                try:
                    s1h, s1r = os.path.getsize(hdr_path), os.path.getsize(raw_path)
                    time.sleep(1.0)
                    s2h, s2r = os.path.getsize(hdr_path), os.path.getsize(raw_path)
                    if not (s1h == s2h and s1r == s2r):
                        continue
                except Exception:
                    continue

                _log(f"Detected new pair: {hdr_path} + {raw_path}")
                try:
                    process_hdr(
                        hdr_path,
                        seg=self.seg,
                        seg_cfg=self.seg_cfg,
                        pipeline_cfg=self.pipeline_cfg,
                        state=self.state,
                        input_folder_name_for_meta=folder_name_for_meta,
                    )
                    self.processed.add(base)
                    _log(f"Processing SUCCESS for {base} → {self.out_dir}")
                except Exception as e:
                    _log(f"Processing FAILED for {base}: {e!r}")

            time.sleep(self.file_poll_interval)

        _log(f"Stop event set; stopping monitor for {self.folder_path}")


# ---------- top-level watcher ----------


class HyperBirdWatcher:
    """
    Continuously watches `watch_dir` for new subfolders and spawns a _FolderMonitor for each.
    No process gating (always on).
    """

    def __init__(
        self,
        watch_dir: str,
        seg,
        seg_cfg,
        pipeline_cfg,
        *,
        folder_poll_interval: float = 1.0,
        file_poll_interval: float = 1.0,
    ):
        self.watch_dir = watch_dir
        self.seg = seg
        self.seg_cfg = seg_cfg
        self.pipeline_cfg = pipeline_cfg

        self.folder_poll_interval = folder_poll_interval
        self.file_poll_interval = file_poll_interval

        self._known_folders = set()
        self._monitors: dict[str, tuple[_FolderMonitor, threading.Event]] = {}
        self._stop_event_global = threading.Event()
        self._thread: Optional[threading.Thread] = None
        self._lock = threading.Lock()

    def _init_known_folders(self):
        if not os.path.isdir(self.watch_dir):
            raise FileNotFoundError(f"WATCH_DIR does not exist: {self.watch_dir}")
        self._known_folders = {
            os.path.join(self.watch_dir, d)
            for d in os.listdir(self.watch_dir)
            if not _ignore_top_level_folder(d)
            and os.path.isdir(os.path.join(self.watch_dir, d))
        }

    def _spawn_monitor_for_folder(self, folder: str):
        stop_evt = threading.Event()
        mon = _FolderMonitor(
            folder_path=folder,
            stop_event=stop_evt,
            seg=self.seg,
            seg_cfg=self.seg_cfg,
            pipeline_cfg=copy.deepcopy(
                self.pipeline_cfg
            ),  # creates an independent copy
            file_poll_interval=self.file_poll_interval,
        )
        self._monitors[folder] = (mon, stop_evt)
        mon.start()

    def _stop_all_monitors(self, join_timeout: float = 5.0):
        with self._lock:
            for _, (_, evt) in list(self._monitors.items()):
                evt.set()
            for folder, (mon, _) in list(self._monitors.items()):
                mon.join(timeout=join_timeout)
                _log(f"Monitor thread joined for {folder}")
                self._monitors.pop(folder, None)

    def start(self):
        if self._thread and self._thread.is_alive():
            return
        self._stop_event_global.clear()
        self._thread = threading.Thread(target=self.run, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop_event_global.set()
        with self._lock:
            for _, (_, evt) in list(self._monitors.items()):
                evt.set()

    def join(self, timeout: Optional[float] = None):
        t = self._thread
        if t:
            t.join(timeout=timeout)

    def run(self):
        try:
            _log(f"Starting folder watchdog. Watching: {self.watch_dir}")
            self._init_known_folders()
        except FileNotFoundError as e:
            _log(f"ERROR: {e}")
            return

        try:
            while not self._stop_event_global.is_set():
                try:
                    current = {
                        os.path.join(self.watch_dir, d)
                        for d in os.listdir(self.watch_dir)
                        if not _ignore_top_level_folder(d)
                        and os.path.isdir(os.path.join(self.watch_dir, d))
                    }
                except Exception as e:
                    _log(f"Error listing watch dir: {e}")
                    current = set()

                added = current - self._known_folders
                removed = self._known_folders - current

                for folder in added:
                    _log(f"New folder detected: {folder}")
                    self._spawn_monitor_for_folder(folder)

                for folder in removed:
                    with self._lock:
                        entry = self._monitors.pop(folder, None)
                    if entry:
                        mon, evt = entry
                        evt.set()
                        mon.join(timeout=2.0)
                        _log(f"Folder removed; stopped monitor for: {folder}")

                self._known_folders = current
                time.sleep(self.folder_poll_interval)

        except KeyboardInterrupt:
            _log("KeyboardInterrupt received. Cleaning up monitors.")
            self._stop_all_monitors(join_timeout=2.0)
        finally:
            _log("Watcher exiting.")

## Configurations

### Pipeline Config

In [7]:
# Pipeline configuration
pipeline_cfg = PipelineCfg(
    temp_root="temp",
    save_ffc=False,
    assume_interleave=None,
    illuminants_mat_path="./src/reference/D_illuminants.mat",
    illuminant_d=65,
    shrink_distance=100,
)

### GSAM Instance

In [8]:
# Build GSAM2 segmenter instance
seg_cfg = SegmenterConfig(
    text_prompt="green, leaf, circle",
    hf_model_id="IDEA-Research/grounding-dino-base",
    detector_device=DEVICE.type,
    sam2_cfg_path="configs/sam2.1/sam2.1_hiera_l.yaml",
    sam2_ckpt_path="../sam2/checkpoints/sam2.1_hiera_large.pt",
    sam2_device=DEVICE.type,
    box_threshold=0.30,
    max_dets=1,
    multimask_output=False,
)

seg = GSAM2_Segmenter(
    hf_model_id=seg_cfg.hf_model_id,
    detector_device=seg_cfg.detector_device,
    sam2_cfg_path=seg_cfg.sam2_cfg_path,
    sam2_ckpt_path=seg_cfg.sam2_ckpt_path,
    sam2_device=seg_cfg.sam2_device,
    box_threshold=seg_cfg.box_threshold,
    max_dets=seg_cfg.max_dets,
    multimask_output=seg_cfg.multimask_output,
)

### Watcher Processing (Pls Run this before runing the hyperbird scanning program!!!) 

In [ ]:
# Input folder to process
watch_dir = "/media/hyperbird/996eaa96-1b30-4b7e-902e-40be9c6c8ae9"  

watcher = HyperBirdWatcher(
    watch_dir=watch_dir,
    seg=seg,
    seg_cfg=seg_cfg,
    pipeline_cfg=   pipeline_cfg,
    folder_poll_interval=1.0,
    file_poll_interval=1.0,
)

try:
    watcher.start()  # Non-blocking: starts the main loop in a thread
    watcher.join()
except KeyboardInterrupt:
    watcher.stop()
    watcher.join()

2026-01-14 16:31:37 - Starting folder watchdog. Watching: /media/hyperbird/996eaa96-1b30-4b7e-902e-40be9c6c8ae9
2026-01-14 16:31:45 - New folder detected: /media/hyperbird/996eaa96-1b30-4b7e-902e-40be9c6c8ae9/test (Copy)
2026-01-14 16:31:45 - Started monitoring folder: /media/hyperbird/996eaa96-1b30-4b7e-902e-40be9c6c8ae9/test (Copy)
2026-01-14 16:31:50 - Detected new pair: /media/hyperbird/996eaa96-1b30-4b7e-902e-40be9c6c8ae9/test (Copy)/001-black.hdr + /media/hyperbird/996eaa96-1b30-4b7e-902e-40be9c6c8ae9/test (Copy)/001-black.raw
2026-01-14 16:32:26 - Processing SUCCESS for 001-black → /media/hyperbird/996eaa96-1b30-4b7e-902e-40be9c6c8ae9/test (Copy)/_processed
